<a href="https://colab.research.google.com/github/2403a52026-lgtm/NLP.LabAssignments/blob/main/NLP_LAB_07_2403a52026_B_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import nltk
import string
import numpy as np
import pandas as pd

from nltk.corpus import stopwords, wordnet
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Download required NLTK data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [2]:
documents = [
    # Sports
    "The football team won the championship match",
    "Cricket players trained hard for the tournament",
    "The athlete broke the world record",
    "The coach planned a new strategy",

    # Politics
    "The government passed a new law",
    "Elections were conducted peacefully",
    "The president addressed the nation",
    "Parliament debated the budget proposal",

    # Health
    "Doctors recommend regular exercise",
    "The hospital introduced new medical equipment",
    "Healthy diet prevents diseases",
    "Vaccination helps in immunity",

    # Technology
    "Artificial intelligence is transforming industries",
    "The software update improved performance",
    "Cybersecurity protects user data",
    "Cloud computing enables scalability",

    # Mixed
    "Technology improves healthcare systems",
    "Sports require discipline and training",
    "Political decisions affect public health",
    "AI assists doctors in diagnosis"
]


In [4]:
nltk.download('punkt_tab')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return " ".join(tokens)

cleaned_docs = [preprocess(doc) for doc in documents]

print(cleaned_docs[:5])

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


['football team championship match', 'cricket player trained hard tournament', 'athlete broke world record', 'coach planned new strategy', 'government passed new law']


In [5]:
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(cleaned_docs)


In [6]:
cosine_sim = cosine_similarity(tfidf_matrix)

cosine_df = pd.DataFrame(cosine_sim, index=documents, columns=documents)
cosine_df.head()


,The football team won the championship match,Cricket players trained hard for the tournament,The athlete broke the world record,The coach planned a new strategy,The government passed a new law,Elections were conducted peacefully,The president addressed the nation,Parliament debated the budget proposal,Doctors recommend regular exercise,The hospital introduced new medical equipment,Healthy diet prevents diseases,Vaccination helps in immunity,Artificial intelligence is transforming industries,The software update improved performance,Cybersecurity protects user data,Cloud computing enables scalability,Technology improves healthcare systems,Sports require discipline and training,Political decisions affect public health,AI assists doctors in diagnosis
The football team won the championship match,1.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Cricket players trained hard for the tournament,0.0,1.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
The athlete broke the world record,0.0,0.0,1.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
The coach planned a new strategy,0.0,0.0,0.0,1.000000,0.173355,0.0,0.0,0.0,0.0,0.153493,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
The government passed a new law,0.0,0.0,0.0,0.173355,1.000000,0.0,0.0,0.0,0.0,0.153493,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [7]:
def jaccard_similarity(doc1, doc2):
    set1 = set(doc1.split())
    set2 = set(doc2.split())
    return len(set1 & set2) / len(set1 | set2)

jaccard_scores = []
for i in range(len(cleaned_docs)):
    for j in range(i+1, len(cleaned_docs)):
        jaccard_scores.append((i, j, jaccard_similarity(cleaned_docs[i], cleaned_docs[j])))

jaccard_scores[:5]


[(0, 1, 0.0), (0, 2, 0.0), (0, 3, 0.0), (0, 4, 0.0), (0, 5, 0.0)]

In [8]:
def wordnet_similarity(word1, word2):
    syn1 = wordnet.synsets(word1)
    syn2 = wordnet.synsets(word2)
    if syn1 and syn2:
        return syn1[0].wup_similarity(syn2[0])
    return None

pairs = [("doctor", "physician"), ("football", "sport"), ("health", "medicine"),
         ("government", "politics"), ("computer", "technology")]

for w1, w2 in pairs:
    print(w1, w2, wordnet_similarity(w1, w2))


doctor physician 1.0
football sport 0.8888888888888888
health medicine 0.18181818181818182
government politics 0.3333333333333333
computer technology 0.1111111111111111
